# 02 - Automated Patch Generation Pipeline

This notebook runs the complete dataset creation pipeline:
1. Unpacks raw LISS-IV scene archives (`.zip`)
2. Normalizes pixel intensities with 2%-98% linear stretch
3. Generates continuous multi-octave synthetic cloud masks (wispy, bank, overcast)
4. Slices scenes into 256x256 paired patches with strict quality filtering
5. Saves paired patches (`clear_XXXXX.npy` and `cloudy_XXXXX.npy`) for cGAN training


In [ ]:
import sys
from pathlib import Path

# Add project root to path for importing scripts
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.cloud_simulator import generate_realistic_cloud_mask, add_synthetic_clouds
from scripts.generate_patches import process_scenes

DATA_DIR = PROJECT_ROOT / "Data"
RAW_ZIPS = DATA_DIR / "Raw_Zips"
TEMP_DIR = DATA_DIR / "temp_extract"
CLEAR_DIR = DATA_DIR / "Clear_Patches"
CLOUDY_DIR = DATA_DIR / "Cloudy_Patches"

print(f"Data directory: {DATA_DIR.resolve()}")


### 1. Test Synthetic Cloud Simulation
Inspect the multi-octave cloud masks and synthetic cloud generation across different cloud formations:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate demo cloud masks
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
types = ["wispy", "bank", "overcast"]

for i, t in enumerate(types):
    mask = generate_realistic_cloud_mask(256, 256, cloud_type=t)
    axes[i].imshow(mask, cmap="Blues_r")
    axes[i].set_title(f"Type: {t.capitalize()}")
    axes[i].axis("off")

plt.suptitle("Multi-Octave Synthetic Cloud Formations", fontsize=14)
plt.tight_layout()
plt.show()


### 2. Run Automated Patch Extraction Pipeline
Extract patches from scenes. You can set `max_patches` to limit the run during testing, or `None` for all.


In [ ]:
# Extraction configuration
PATCH_SIZE = 256
STRIDE = 128
VALID_THRESHOLD = 0.90
MIN_STD_DEV = 0.05
MIN_MEAN_BRIGHTNESS = 0.15

total_saved, total_skipped = process_scenes(
    raw_zips_dir=RAW_ZIPS,
    temp_extract_dir=TEMP_DIR,
    clear_dir=CLEAR_DIR,
    cloudy_dir=CLOUDY_DIR,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    valid_threshold=VALID_THRESHOLD,
    min_std=MIN_STD_DEV,
    min_mean=MIN_MEAN_BRIGHTNESS
)

print(f"Pipeline finished! Total patches in dataset: {total_saved}")
